# Computational Benchmarks: JAXOperator vs SciPySparseOperator

This notebook quantifies the memory and runtime advantages of the Phase 2
`JAXOperator` (compact trace storage + JAX JIT) over the Phase 1
`SciPySparseOperator` (full CSR sparse matrix), and compares convergence
of `JAXProximalSolver` (FISTA) against `SpectralSolver` (LSQR).

**Audience:** developers and contributors evaluating Phase 2 design choices.

In [ ]:
from __future__ import annotations

import time
import timeit
import tracemalloc
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import spectrex
from spectrex import (
    EigenspectraBasis,
    InstrumentConfig,
    JAXOperator,
    JAXProximalSolver,
    NoiseModel,
    SciPySparseOperator,
    SpectralSolver,
)

warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path.cwd()
REPO = NOTEBOOK_DIR.parent
TESTDATA = REPO / 'testdata'
print(f'spectrex version: {spectrex.__version__}')


## Instrument Setup

In [ ]:
config = InstrumentConfig.from_files(
    conf_path=TESTDATA / 'Config Files' / 'GR150R.F150W.220725.conf',
    wavelengthrange_path=TESTDATA / 'jwst_niriss_wavelengthrange_0002.asdf',
    sensitivity_dir=TESTDATA / 'SenseConfig' / 'wfss-grism-configuration',
    filter_name='F150W',
    n_wavelengths=150,
)
basis = EigenspectraBasis.from_csv(
    TESTDATA / 'eigenspectra_kurucz.csv',
    config.wavelengths,
)

IMAGE_SHAPE = (50, 20)
N_ROWS, N_COLS = IMAGE_SHAPE
N_PIX = N_ROWS * N_COLS
M = basis.n_components
NOISE_MODEL = NoiseModel(read_noise=5.0)
RNG = np.random.default_rng(2026)

print(f'Image shape: {IMAGE_SHAPE}, n_pix={N_PIX}')
print(f'Basis components M={M}')
print(f'n_wavelengths={len(config.wavelengths)}')
print(f'n_orders={len(config.orders)}')


## Section 1: Memory Footprint

### Analytical Model

**`SciPySparseOperator`** stores a CSR sparse matrix of shape `(N_pix, K×M)`.
- `data` array: `nnz` float64 values = `K × n_orders × n_lambda × 8` bytes
- `indices` array: `nnz` int32 values = `K × n_orders × n_lambda × 4` bytes
- `indptr` array: `(N_pix + 1)` int32 values = `(N_pix + 1) × 4` bytes

**Key insight:** the `indptr` array alone scales with `N_pix`, independently of K.
For NIRISS full-frame (2048×2048), `N_pix = 4,194,304` → `indptr ≈ 16 MB` per operator.

**`JAXOperator`** stores:
- `trace_indices[K, n_orders, n_lambda]` int32: `K × n_orders × n_lambda × 4` bytes
- `weights[n_orders, n_lambda, M]` float32: `n_orders × n_lambda × M × 4` bytes

**Key insight:** weights are shared across all K sources — factor-K saving vs CSR data.
Memory is independent of `N_pix`.

**Analytical comparison at full NIRISS scale (K=100, n_orders=3, n_lambda=150, M=10):**
| Component | SciPySparse | JAXOperator |
|-----------|-------------|-------------|
| data/indices | K×O×L×12 ≈ 5.4 MB | K×O×L×4 ≈ 1.8 MB |
| weights | K×O×L×M×8 ≈ 36 MB | O×L×M×4 ≈ 0.18 MB |
| indptr | (N_pix+1)×4 ≈ 16 MB | — |
| **Total** | **~57 MB** | **~2 MB** |

### Measured Memory vs K

In [ ]:
K_GRID = [1, 2, 5, 10, 20, 50]

mem_scipy_mb = []
mem_jax_mb   = []

for k in K_GRID:
    src_pos = np.column_stack([
        RNG.integers(0, N_ROWS, size=k),
        RNG.integers(0, N_COLS, size=k),
    ]).astype(np.float64)

    scipy_op = SciPySparseOperator.build(config, basis, IMAGE_SHAPE)
    jax_op   = JAXOperator.build(config, basis, IMAGE_SHAPE, src_pos)

    # SciPy CSR memory
    m_scipy = (
        scipy_op._matrix.data.nbytes
        + scipy_op._matrix.indices.nbytes
        + scipy_op._matrix.indptr.nbytes
    ) / 1e6

    # JAX compact memory
    m_jax = (
        np.asarray(jax_op._trace_indices).nbytes
        + np.asarray(jax_op._weights).nbytes
    ) / 1e6

    mem_scipy_mb.append(m_scipy)
    mem_jax_mb.append(m_jax)
    print(f'K={k:2d}: SciPy {m_scipy:.3f} MB  JAX {m_jax:.3f} MB')


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(K_GRID, mem_scipy_mb, 'o-', color='steelblue', label='SciPySparseOperator (CSR)', lw=2)
ax.plot(K_GRID, mem_jax_mb,   's-', color='tomato',    label='JAXOperator (compact trace)', lw=2)
ax.set_xlabel('Number of sources (K)')
ax.set_ylabel('Memory (MB)')
ax.set_title('Operator Memory Footprint vs K')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('\nNote: SciPy indptr is constant — reflects N_pix overhead independent of K.')


## Section 2: Runtime Benchmarks

Measurements cover:
- **Build time**: constructing the operator from instrument config.
- **Apply time**: single forward pass `op.apply(a)` — JAX first call (includes JIT
  compilation) vs steady-state (subsequent calls).
- **Adjoint time**: single `op.apply_adjoint(f)` — same JIT distinction for JAX.

Timings use `timeit.timeit` with `number=3` repeats, reporting mean wall-clock seconds.
JAX steady-state is measured from the 2nd call onward.

In [ ]:
BENCH_K_GRID = [1, 2, 5, 10, 20]
N_REPEATS = 3

bench = {
    'k': [],
    't_build_scipy': [], 't_build_jax': [],
    't_apply_scipy': [], 't_apply_jax_cold': [], 't_apply_jax_warm': [],
    't_adjoint_scipy': [], 't_adjoint_jax_cold': [], 't_adjoint_jax_warm': [],
}

for k in BENCH_K_GRID:
    src_pos = np.column_stack([
        RNG.integers(0, N_ROWS, size=k),
        RNG.integers(0, N_COLS, size=k),
    ]).astype(np.float64)

    # Build times
    t_bs = timeit.timeit(
        lambda: SciPySparseOperator.build(config, basis, IMAGE_SHAPE),
        number=N_REPEATS
    ) / N_REPEATS
    t_bj = timeit.timeit(
        lambda: JAXOperator.build(config, basis, IMAGE_SHAPE, src_pos),
        number=N_REPEATS
    ) / N_REPEATS

    # Build once for apply/adjoint benchmarks
    scipy_op = SciPySparseOperator.build(config, basis, IMAGE_SHAPE)
    jax_op   = JAXOperator.build(config, basis, IMAGE_SHAPE, src_pos)

    a_scipy = np.random.default_rng(0).standard_normal(scipy_op.n_coefficients)
    a_jax   = np.random.default_rng(0).standard_normal(jax_op.n_coefficients)
    f_vec   = np.random.default_rng(0).standard_normal(N_PIX)

    # Apply times — scipy
    t_as = timeit.timeit(lambda: scipy_op.apply(a_scipy), number=N_REPEATS) / N_REPEATS
    t_ds = timeit.timeit(lambda: scipy_op.apply_adjoint(f_vec), number=N_REPEATS) / N_REPEATS

    # Apply times — JAX cold (first call triggers JIT)
    t0 = time.perf_counter(); jax_op.apply(a_jax); t_aj_cold = time.perf_counter() - t0
    t0 = time.perf_counter(); jax_op.apply_adjoint(f_vec); t_dj_cold = time.perf_counter() - t0

    # Apply times — JAX warm (steady-state)
    t_aj_warm = timeit.timeit(lambda: jax_op.apply(a_jax), number=N_REPEATS) / N_REPEATS
    t_dj_warm = timeit.timeit(lambda: jax_op.apply_adjoint(f_vec), number=N_REPEATS) / N_REPEATS

    bench['k'].append(k)
    bench['t_build_scipy'].append(t_bs)
    bench['t_build_jax'].append(t_bj)
    bench['t_apply_scipy'].append(t_as)
    bench['t_apply_jax_cold'].append(t_aj_cold)
    bench['t_apply_jax_warm'].append(t_aj_warm)
    bench['t_adjoint_scipy'].append(t_ds)
    bench['t_adjoint_jax_cold'].append(t_dj_cold)
    bench['t_adjoint_jax_warm'].append(t_dj_warm)
    print(f'K={k:2d}: build scipy={t_bs:.3f}s jax={t_bj:.3f}s | '
          f'apply scipy={t_as*1e3:.1f}ms jax_warm={t_aj_warm*1e3:.1f}ms')


### Runtime Summary Table

In [ ]:
ks = bench['k']
print(f"{'K':>4} | {'Build SciPy':>12} {'Build JAX':>10} | "
      f"{'Apply SciPy':>12} {'Apply JAX cold':>15} {'Apply JAX warm':>15} |"
      f"{'Adj SciPy':>10} {'Adj JAX warm':>13}")
print('-' * 105)
for i, k in enumerate(ks):
    print(
        f'{k:>4} | '
        f'{bench["t_build_scipy"][i]*1e3:>10.1f}ms '
        f'{bench["t_build_jax"][i]*1e3:>10.1f}ms | '
        f'{bench["t_apply_scipy"][i]*1e3:>10.1f}ms '
        f'{bench["t_apply_jax_cold"][i]*1e3:>13.1f}ms '
        f'{bench["t_apply_jax_warm"][i]*1e3:>13.1f}ms | '
        f'{bench["t_adjoint_scipy"][i]*1e3:>10.1f}ms '
        f'{bench["t_adjoint_jax_warm"][i]*1e3:>10.1f}ms'
    )


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Build time
ax1.plot(ks, [t*1e3 for t in bench['t_build_scipy']], 'o-', color='steelblue', label='SciPySparse', lw=2)
ax1.plot(ks, [t*1e3 for t in bench['t_build_jax']],   's-', color='tomato',    label='JAXOperator', lw=2)
ax1.set_xlabel('K (sources)')
ax1.set_ylabel('Time (ms)')
ax1.set_title('Operator Build Time vs K')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Apply time
ax2.plot(ks, [t*1e3 for t in bench['t_apply_scipy']],    'o-', color='steelblue', label='SciPy apply', lw=2)
ax2.plot(ks, [t*1e3 for t in bench['t_apply_jax_cold']], 'v--', color='tomato',   label='JAX apply (cold/JIT)', lw=1.5, alpha=0.7)
ax2.plot(ks, [t*1e3 for t in bench['t_apply_jax_warm']], 's-', color='tomato',    label='JAX apply (warm)', lw=2)
ax2.set_xlabel('K (sources)')
ax2.set_ylabel('Time (ms)')
ax2.set_title('Forward Apply Time vs K')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Section 3: Solve Time and Convergence

Fixed problem: K=5 sources, same `IMAGE_SHAPE=(50,20)` scene.

- **SpectralSolver (LSQR)**: solved with `scipy.sparse.linalg.lsqr`; residual
  norm tracked every 10 iterations via the `iter_lim` parameter.
- **JAXProximalSolver (FISTA)**: FISTA with group-L1; residual tracked every
  iteration. First call (JIT compile) excluded from steady-state timing.

The plot shows residual norm vs cumulative wall-clock time, illustrating the
JIT amortisation break-even point.

In [ ]:
import scipy.sparse.linalg as spla

K_CONV = 5
src_pos_conv = np.column_stack([
    RNG.integers(0, N_ROWS, size=K_CONV),
    RNG.integers(0, N_COLS, size=K_CONV),
]).astype(np.float64)

scipy_op_conv = SciPySparseOperator.build(config, basis, IMAGE_SHAPE)
jax_op_conv   = JAXOperator.build(config, basis, IMAGE_SHAPE, src_pos_conv)

# Ground truth and noisy observation
a_true_conv = RNG.standard_normal(K_CONV * M)
f_clean_conv = jax_op_conv.apply(a_true_conv).reshape(IMAGE_SHAPE)
f_noisy_conv = NOISE_MODEL.sample(f_clean_conv, np.random.default_rng(99))
f_flat_conv  = f_noisy_conv.ravel()

# Precision weights
w = NOISE_MODEL.precision_weights(f_noisy_conv).ravel()

# Support mask (SpectralSolver)
src_flat_conv = [
    int(round(r)) * N_COLS + int(round(c))
    for r, c in src_pos_conv
]
mask_conv = np.zeros(N_PIX * M, dtype=bool)
for p in src_flat_conv:
    mask_conv[p * M : (p + 1) * M] = True

print('Setup done.')


### LSQR Convergence (checkpoint every 10 iterations)

In [ ]:
# Build weighted system A = diag(w) @ H, b = diag(w) @ f
H = scipy_op_conv._matrix[:, mask_conv]  # restrict to support
W = w
WH = H.multiply(W[:, None])   # element-wise row scaling
Wf = W * f_flat_conv

MAX_LSQR_ITER = 300
CHUNK = 10
lsqr_times = []
lsqr_residuals = []
x_lsqr = np.zeros(H.shape[1])
t_lsqr_start = time.perf_counter()

for chunk_start in range(0, MAX_LSQR_ITER, CHUNK):
    result = spla.lsqr(
        WH, Wf,
        x0=x_lsqr,
        iter_lim=CHUNK,
        atol=0, btol=0, conlim=0,
    )
    x_lsqr = result[0]
    r_norm = float(np.linalg.norm(Wf - WH @ x_lsqr))
    elapsed = time.perf_counter() - t_lsqr_start
    lsqr_times.append(elapsed)
    lsqr_residuals.append(r_norm)

print(f'LSQR: {MAX_LSQR_ITER} iters, final residual={lsqr_residuals[-1]:.4f}')


### FISTA Convergence (per-iteration residual)

In [ ]:
# FISTA warm-up (JIT compile)
_ = JAXProximalSolver(
    jax_op_conv, noise_model=NOISE_MODEL, lam=0.05, n_iter=1
).solve(f_noisy_conv)

# FISTA with per-iteration residual tracking
import jax.numpy as jnp
from spectrex.jax_solver import group_soft_threshold, power_iteration

N_FISTA_ITER = 200
LAM = 0.05

# power_iteration expects flat 1D precision_weights
L = power_iteration(jax_op_conv, w, N_PIX, n_iter=30)
step = 1.0 / L

# Manual FISTA loop with timing
x = np.zeros(K_CONV * M)
y = x.copy()
t_fista_iter = 1.0

fista_times = []
fista_residuals = []
t_fista_start = time.perf_counter()

for i in range(1, N_FISTA_ITER + 1):
    # Gradient: H^T W^2 (Hx - f)
    Hx = jax_op_conv.apply(y).ravel()
    grad = jax_op_conv.apply_adjoint(((Hx - f_flat_conv) * w**2).reshape(IMAGE_SHAPE))
    x_new = group_soft_threshold(y - step * grad, LAM * step, K_CONV, M)
    t_new = (1 + np.sqrt(1 + 4 * t_fista_iter**2)) / 2
    y = x_new + ((t_fista_iter - 1) / t_new) * (x_new - x)
    x, t_fista_iter = x_new, t_new
    r = float(np.linalg.norm(jax_op_conv.apply(x).ravel() - f_flat_conv))
    elapsed = time.perf_counter() - t_fista_start
    fista_times.append(elapsed)
    fista_residuals.append(r)

print(f'FISTA: {N_FISTA_ITER} iters, final residual={fista_residuals[-1]:.4f}')


### Residual Norm vs Wall-Clock Time

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(lsqr_times, lsqr_residuals, 'o-', color='steelblue',
            label='LSQR (checkpoints every 10 iters)', lw=2, markersize=4)
ax.semilogy(fista_times, fista_residuals, '-', color='tomato',
            label='FISTA (per iteration, post-JIT)', lw=1.5, alpha=0.9)
ax.set_xlabel('Wall-clock time (s)')
ax.set_ylabel('Residual norm (log scale)')
ax.set_title('Convergence: LSQR vs FISTA (K=5 sources)')
ax.legend()
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()
print(f'\nJIT amortisation: FISTA warm-up run excluded from timing above.')
print(f'For {N_FISTA_ITER} full solves, JIT overhead is negligible.')


## Summary

| Metric | SciPySparseOperator / SpectralSolver | JAXOperator / JAXProximalSolver |
|--------|--------------------------------------|----------------------------------|
| Memory | O(K×O×L + **N_pix**) — N_pix overhead | O(K×O×L + O×L×M) — **N_pix-free** |
| Weights stored | K × O × L × M (per-source) | O × L × M (**shared**) |
| Apply (warm) | CSR matvec | JIT-compiled einsum + scatter |
| Solver | LSQR (unconstrained) | FISTA group-L1 (sparse, per-source) |

**When to use which:**
- `SciPySparseOperator` + `SpectralSolver`: small images, CPU-only, no JAX, or fast
  prototyping with very few sources.
- `JAXOperator` + `JAXProximalSolver`: large images, many sources, GPU-available,
  or whenever accuracy in crowded fields matters.